# SASRec. Квантование с обучением (QAT) и оценка на CPU

За основу возьмем модель из оригинального репозитория https://github.com/pmixer/SASRec.pytorch/blob/main/python/model.py  
Данные для обучения возьмем из того же репозитория https://github.com/pmixer/SASRec.pytorch/blob/main/python/data/ml-1m.txt

In [2]:
# !mkdir data
# !wget https://raw.githubusercontent.com/pmixer/SASRec.pytorch/refs/heads/main/python/data/ml-1m.txt
# !mv ml-1m.txt data/.

In [ ]:
import os
import torch

from torch import nn
from pathlib import Path
from typing import Any, Dict
from train_functions import train_fp32, train_qat, apply_adaround

from models.quantization import QuantSASRec
from data.dataloader import create_dataloaders
from utils import load_config, set_random_seeds, ensure_dir

In [4]:
class Args:
    """Helper to pass config parameters to SASRec model."""
    def __init__(self, config: Dict[str, Any]):
        self.hidden_units = config["model"]["hidden_units"]
        self.num_blocks = config["model"]["num_blocks"]
        self.num_heads = config["model"]["num_heads"]
        self.dropout_rate = config["model"]["dropout_rate"]
        self.maxlen = config["model"]["maxlen"]
        self.device = torch.device(config["experiment"].get("device", "cuda" if torch.cuda.is_available() else "cpu"))
        self.norm_first = config["model"].get("norm_first", False)

In [ ]:
def start_train(config_name):
    config_path = os.path.join("./configs", config_name)
    config = load_config(config_path)

    set_random_seeds(config["experiment"].get("seed", 42))
    args = Args(config)
    
    train_loader, _, _, dataset = create_dataloaders(
            config, seed=config["experiment"].get("seed", 42))
    
    _, _, _, usernum, itemnum = dataset
    model = QuantSASRec(usernum, itemnum, args).to(args.device)
    
    quant_cfg = config.get("quantization", {})
    strategy_name = quant_cfg.get("method", "fp32").lower()
    strategy_name = quant_cfg.get("method", "fp32").lower()
    checkpoint_dir = Path(config.get("paths", {}).get("checkpoints_dir", "./checkpoints"))
    run_name = config["experiment"].get("run_name", "sasrec_run")
    checkpoint_subdir = checkpoint_dir / run_name
    ensure_dir(checkpoint_subdir)
    results_dir = Path(config.get("paths", {}).get("results_dir", "./results"))
    ensure_dir(results_dir)

    logger = None
    criterion = nn.BCEWithLogitsLoss()

    if strategy_name == "fp32":
        train_fp32(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_fp32.pth",
            logger=logger
        )  
    elif strategy_name in ["lsq", "apot", "qdrop"]:
        train_qat(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            strategy_name=strategy_name,
            quant_config=quant_cfg,
            checkpoint_dir=str(checkpoint_subdir),
            save_name=f"sasrec_{strategy_name}.pth",
            logger=logger
        )
    elif strategy_name == "adaround":
        fp32_epochs = config["training"].get("fp32_epochs", 0)
        fp32_ckpt = quant_cfg.get("base_checkpoint", None)
        
        fp32_ckpt_name = "sasrec_fp32_for_adaround.pth"
        
        if fp32_epochs > 0:
             train_fp32(
                model=model,
                train_loader=train_loader,
                dataset=dataset,
                config=config,
                criterion=criterion,
                args=args,
                checkpoint_dir=str(checkpoint_subdir),
                save_name=fp32_ckpt_name,
                logger=logger
            )
        elif fp32_ckpt:
             fp32_ckpt_path = Path(fp32_ckpt)
             if not fp32_ckpt_path.exists():
                 fp32_ckpt_path = checkpoint_subdir / fp32_ckpt
             
             if not fp32_ckpt_path.exists():
                  raise FileNotFoundError(f"Base checkpoint not found: {fp32_ckpt}")
             if fp32_ckpt_path.parent == checkpoint_subdir:
                 fp32_ckpt_name = fp32_ckpt_path.name
             else:
                 fp32_ckpt_name = str(fp32_ckpt_path)
        
        apply_adaround(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            args=args,
            fp32_checkpoint=fp32_ckpt_name,
            adaround_config=quant_cfg,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_adaround.pth",
            logger=logger
        )

In [6]:
start_train('fp32.yml')

Loading data from ./data/ml-1m.txt...
Starting FP32 training...


[FP32] Epoch 1/20 | Loss: 6.2533 | NDCG@10: 0.0005 | Hit@10: 0.0012
Saved best FP32 checkpoint (NDCG: 0.0005)


[FP32] Epoch 2/20 | Loss: 5.3796 | NDCG@10: 0.0013 | Hit@10: 0.0030
Saved best FP32 checkpoint (NDCG: 0.0013)


[FP32] Epoch 3/20 | Loss: 3.7744 | NDCG@10: 0.0016 | Hit@10: 0.0045
Saved best FP32 checkpoint (NDCG: 0.0016)


[FP32] Epoch 4/20 | Loss: 2.6201 | NDCG@10: 0.0022 | Hit@10: 0.0046
Saved best FP32 checkpoint (NDCG: 0.0022)


...

[FP32] Epoch 17/20 | Loss: 0.9406 | NDCG@10: 0.0119 | Hit@10: 0.0255
Saved best FP32 checkpoint (NDCG: 0.0119)


[FP32] Epoch 18/20 | Loss: 0.9392 | NDCG@10: 0.0124 | Hit@10: 0.0270
Saved best FP32 checkpoint (NDCG: 0.0124)


[FP32] Epoch 19/20 | Loss: 0.9390 | NDCG@10: 0.0117 | Hit@10: 0.0248


[FP32] Epoch 20/20 | Loss: 0.9357 | NDCG@10: 0.0113 | Hit@10: 0.0240
FP32 training done. Best NDCG (Val): 0.0124
Loading best FP32 checkpoint from checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth for final testing...
Running final test evaluation (user_test split)...


[FP32][Test] NDCG@10: 0.0107 | Hit@10: 0.0219


In [7]:
start_train('lsq.yml')

Loading data from ./data/ml-1m.txt...
Starting QAT with LSQ...
Running dummy pass to initialize quantization parameters...


[QAT lsq] Epoch 1/20 | Loss: 6.2115 | NDCG@10: 0.0006 | Hit@10: 0.0015
Saved best QAT checkpoint (NDCG: 0.0006)


[QAT lsq] Epoch 2/20 | Loss: 5.8563 | NDCG@10: 0.0006 | Hit@10: 0.0017
Saved best QAT checkpoint (NDCG: 0.0006)


[QAT lsq] Epoch 3/20 | Loss: 5.5092 | NDCG@10: 0.0007 | Hit@10: 0.0018
Saved best QAT checkpoint (NDCG: 0.0007)


[QAT lsq] Epoch 4/20 | Loss: 5.1482 | NDCG@10: 0.0006 | Hit@10: 0.0017


...

[QAT lsq] Epoch 17/20 | Loss: 1.7666 | NDCG@10: 0.0015 | Hit@10: 0.0040
Saved best QAT checkpoint (NDCG: 0.0015)


[QAT lsq] Epoch 18/20 | Loss: 1.6049 | NDCG@10: 0.0017 | Hit@10: 0.0043
Saved best QAT checkpoint (NDCG: 0.0017)


[QAT lsq] Epoch 19/20 | Loss: 1.4729 | NDCG@10: 0.0020 | Hit@10: 0.0051
Saved best QAT checkpoint (NDCG: 0.0020)


[QAT lsq] Epoch 20/20 | Loss: 1.3744 | NDCG@10: 0.0024 | Hit@10: 0.0063
Saved best QAT checkpoint (NDCG: 0.0024)
QAT (lsq) done. Best NDCG (Val): 0.0024
Loading best QAT checkpoint from checkpoints/sasrec_runs/sasrec_lsq_run/sasrec_lsq.pth for final testing...
Running final test evaluation (user_test split)...


[QAT lsq][Test] NDCG@10: 0.0037 | Hit@10: 0.0096


In [8]:
start_train('lsq2.yml')

Loading data from ./data/ml-1m.txt...
Starting QAT with LSQ...
Running dummy pass to initialize quantization parameters...


[QAT lsq] Epoch 1/20 | Loss: 6.3167 | NDCG@10: 0.0005 | Hit@10: 0.0012
Saved best QAT checkpoint (NDCG: 0.0005)


[QAT lsq] Epoch 2/20 | Loss: 5.8811 | NDCG@10: 0.0005 | Hit@10: 0.0013
Saved best QAT checkpoint (NDCG: 0.0005)


[QAT lsq] Epoch 3/20 | Loss: 5.4200 | NDCG@10: 0.0006 | Hit@10: 0.0017
Saved best QAT checkpoint (NDCG: 0.0006)


[QAT lsq] Epoch 4/20 | Loss: 5.1426 | NDCG@10: 0.0006 | Hit@10: 0.0017


...

[QAT lsq] Epoch 17/20 | Loss: 1.8743 | NDCG@10: 0.0010 | Hit@10: 0.0028
Saved best QAT checkpoint (NDCG: 0.0010)


[QAT lsq] Epoch 18/20 | Loss: 1.7092 | NDCG@10: 0.0011 | Hit@10: 0.0030
Saved best QAT checkpoint (NDCG: 0.0011)


[QAT lsq] Epoch 19/20 | Loss: 1.5536 | NDCG@10: 0.0013 | Hit@10: 0.0033
Saved best QAT checkpoint (NDCG: 0.0013)


[QAT lsq] Epoch 20/20 | Loss: 1.4472 | NDCG@10: 0.0018 | Hit@10: 0.0045
Saved best QAT checkpoint (NDCG: 0.0018)
QAT (lsq) done. Best NDCG (Val): 0.0018
Loading best QAT checkpoint from checkpoints/sasrec_runs/sasrec_lsq/sasrec_lsq.pth for final testing...
Running final test evaluation (user_test split)...


[QAT lsq][Test] NDCG@10: 0.0027 | Hit@10: 0.0068


In [10]:
start_train('qdrop.yml')

Loading data from ./data/ml-1m.txt...
Starting QAT with QDROP...
Running dummy pass to initialize quantization parameters...


[QAT qdrop] Epoch 1/20 | Loss: 6.2565 | NDCG@10: 0.0005 | Hit@10: 0.0012
Saved best QAT checkpoint (NDCG: 0.0005)


[QAT qdrop] Epoch 2/20 | Loss: 5.4203 | NDCG@10: 0.0013 | Hit@10: 0.0036
Saved best QAT checkpoint (NDCG: 0.0013)


[QAT qdrop] Epoch 3/20 | Loss: 3.8185 | NDCG@10: 0.0019 | Hit@10: 0.0048
Saved best QAT checkpoint (NDCG: 0.0019)


[QAT qdrop] Epoch 4/20 | Loss: 2.6576 | NDCG@10: 0.0025 | Hit@10: 0.0050
Saved best QAT checkpoint (NDCG: 0.0025)


...

[QAT qdrop] Epoch 17/20 | Loss: 0.9400 | NDCG@10: 0.0117 | Hit@10: 0.0255


[QAT qdrop] Epoch 18/20 | Loss: 0.9413 | NDCG@10: 0.0112 | Hit@10: 0.0248


[QAT qdrop] Epoch 19/20 | Loss: 0.9401 | NDCG@10: 0.0131 | Hit@10: 0.0280
Saved best QAT checkpoint (NDCG: 0.0131)


[QAT qdrop] Epoch 20/20 | Loss: 0.9386 | NDCG@10: 0.0119 | Hit@10: 0.0250
QAT (qdrop) done. Best NDCG (Val): 0.0131
Loading best QAT checkpoint from checkpoints/sasrec_runs/sasrec_qdrop/sasrec_qdrop.pth for final testing...
Running final test evaluation (user_test split)...


[QAT qdrop][Test] NDCG@10: 0.0105 | Hit@10: 0.0212


In [11]:
start_train('apot.yml')

Loading data from ./data/ml-1m.txt...
Starting QAT with APOT...
Running dummy pass to initialize quantization parameters...


[QAT apot] Epoch 1/20 | Loss: 6.2717 | NDCG@10: 0.0005 | Hit@10: 0.0013
Saved best QAT checkpoint (NDCG: 0.0005)


[QAT apot] Epoch 2/20 | Loss: 5.5040 | NDCG@10: 0.0011 | Hit@10: 0.0030
Saved best QAT checkpoint (NDCG: 0.0011)


[QAT apot] Epoch 3/20 | Loss: 3.9513 | NDCG@10: 0.0013 | Hit@10: 0.0036
Saved best QAT checkpoint (NDCG: 0.0013)


[QAT apot] Epoch 4/20 | Loss: 2.7578 | NDCG@10: 0.0024 | Hit@10: 0.0050
Saved best QAT checkpoint (NDCG: 0.0024)


...

[QAT apot] Epoch 17/20 | Loss: 0.9497 | NDCG@10: 0.0116 | Hit@10: 0.0258
Saved best QAT checkpoint (NDCG: 0.0116)


[QAT apot] Epoch 18/20 | Loss: 0.9482 | NDCG@10: 0.0114 | Hit@10: 0.0248


[QAT apot] Epoch 19/20 | Loss: 0.9471 | NDCG@10: 0.0108 | Hit@10: 0.0242


[QAT apot] Epoch 20/20 | Loss: 0.9459 | NDCG@10: 0.0108 | Hit@10: 0.0233
QAT (apot) done. Best NDCG (Val): 0.0116
Loading best QAT checkpoint from checkpoints/sasrec_runs/sasrec_apot/sasrec_apot.pth for final testing...
Running final test evaluation (user_test split)...


[QAT apot][Test] NDCG@10: 0.0093 | Hit@10: 0.0200


In [6]:
start_train('adaround.yml')

Loading data from ./data/ml-1m.txt...
Starting AdaRound PTQ...
checkpoints/sasrec_runs/sasrec_adaround
checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth
Loading FP32 checkpoint: checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth
Running AdaRound calibration...
Validating AdaRound model on validation split...


[AdaRound][Val] NDCG@10: 0.0124 | Hit@10: 0.0268
Running final test evaluation for AdaRound model (user_test split)...


[AdaRound][Test] NDCG@10: 0.0108 | Hit@10: 0.0220
AdaRound done. Model saved as sasrec_adaround.pth


In [15]:
!du -s checkpoints/sasrec_runs/*

1428	checkpoints/sasrec_runs/sasrec_adaround
3660	checkpoints/sasrec_runs/sasrec_apot
3652	checkpoints/sasrec_runs/sasrec_fp32
3680	checkpoints/sasrec_runs/sasrec_lsq
3728	checkpoints/sasrec_runs/sasrec_lsq_run
3664	checkpoints/sasrec_runs/sasrec_qdrop
